# An example of reading non-standard data using pandas and xarray

It is very convenient when data is available in a format like netCDF, because it can be easily loaded using `xarray.open_dataset`, particularly when there is also sufficient metadata so the data can be identified and used correctly. 

Sometimes data is only available in a non-standard form which takes some work to put into a convenient form like an `xarray.Dataset`. The purpose of this blog post is step through an example of the process of importing just such a dataset, transforming it into an easy to use `xarray.Dataset`, and then saving it as a netCDF formatted file for easy and convenient re-use.

## Importing the Eastern Australia and New Zealand Drought Atlas (ANZDA) from NCEI, NOAA

About the data: The data is a reconstruction of the Palmer drought sensitivity index across eastern Australia, and is the first Southern Hemisphere gridded drought atlas extending back to CE 1500.

The data is available from this NOAA website:

https://www.ncei.noaa.gov/access/paleo-search/study/20245

which explains the provenance of the data, has extensive meta-data in a number of formats, and a series of data files available for download

The data and grid information are stored in seprate files. Looking first at the grid, it is available in a text file (`anzda-pdsi-xy.txt`) as lat-lon pairs that correspond to each grid cell

The file is tab-delimited, and formatted into columns, the first column is longitude, the second latitude

```
168.25  -46.75  697     87      scPDSI.dat.
169.25  -46.25  699     88      scPDSI.dat.
169.75  -46.25  700     88      scPDSI.dat.
167.75  -45.75  696     89      scPDSI.dat.
168.25  -45.75  697     89      scPDSI.dat.
168.75  -45.75  698     89      scPDSI.dat.
169.25  -45.75  699     89      scPDSI.dat.
169.75  -45.75  700     89      scPDSI.dat.
```

As the data files are tab-separated columnar text files the `pandas` library is a good choice for reading in the data and manipulating it. `pandas` is specifically designed for manipulating tabular data, and much of the machinery of `xarray` is based on `pandas`, but extended to n-dimensions.

In [ ]:
from datetime import datetime
import urllib

import numpy as np
import pandas as pd
import xarray as xr

Use the [pandas.read_csv](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_csv.html) function to read the first two columns directly from the `urlopen` function, with the web address of the data file as a it's argument, returning a [pandas.Dataframe](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html)

Note that the delimiter is specifed to be be tab (`\t`), no header line is specified and the columns are assigned useful names.

In [ ]:
url = "https://www.ncei.noaa.gov/pub/data/paleo/treering/reconstructions/australia/palmer2015pdsi/anzda-pdsi-xy.txt"
xy = pd.read_csv(urllib.request.urlopen(url), 
                 delimiter='\t', 
                 header = None,
                 names = ('lon', 'lat'),
                 usecols = [0, 1])

The `head` method is a useful way to get a small sample of the data to check it

In [ ]:
xy.head()

Plotting is also a great way to quickly check the data. Like `xarray`, `pandas` has a built in [plot method](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.plot.html) that calls out to another library to do the plotting. In this case the default is `matplotlib`.

In [ ]:
xy.plot(x='lon', y='lat', kind='scatter');

Now get the actual data: it is also tab-separated columnar ASCII format, according to the dataset <a href="https://www.ncei.noaa.gov/pub/data/paleo/treering/reconstructions/australia/palmer2015pdsi/readme-palmer2015anzda.txt" target="_blank">README</a>. That is, one reconstruction per column, with the grid cell number as the column header, and rows are by year with the first column the year index. 

As before use the [pandas.read_csv](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_csv.html) function to read the data in, specifying that the delimiter is a tab and that the first column should be treated as the row index. In this case the first line contains useful header information, so that is retained.

In [ ]:
url = "https://www.ncei.noaa.gov/pub/data/paleo/treering/reconstructions/australia/palmer2015pdsi/anzda-recon.txt"
data = pd.read_csv(urllib.request.urlopen(url), delimiter='\t', index_col=0)
data

The data is stored as a matrix with associated time and location vectors, so we need to convert from a "wide" to "long" format using the `melt` method, with the result that there is a row for every unique combination of year and location.

Specifying `ignore_index=False` means it isn't included in the value column, but is retained as an index. This method will create two new columns with the cell index in one, and the values from the table in other. The function has default names `variable` and `value`, but better to give them useful names: `cellref` and `pdsi` respectively.                                                                                              

In [ ]:
data_long = data.melt(ignore_index=False, var_name = 'cellref', value_name='pdsi')
data_long

Next is to add the `lat` and `lon` values for each cell index to the data table above. One way to achieve this is to add a `cellref` column to the `xy` table so it can be merged with `data_long` using the `cellref` variable as a key

In [ ]:
xy['cellref'] = range(1, len(xy)+1, 1)
xy

Now try [merge](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html)

In [ ]:
data_long.merge(xy, on='cellref')

The `merge` fails with an error message 

`You are trying to merge on object and int64 columns`. 

Inspecting the types (`dtypes`) of the two tables shows the issue, the `cellref` column in `data_long` has a type `object`

In [ ]:
xy.dtypes

In [ ]:
data_long.dtypes

The solution is to replace the `cellref` column with the same data converted to a number, and try merging again.

In [ ]:
data_long['cellref'] = pd.to_numeric(data_long['cellref'])
data_long

Note that `reset_index` converts the `Year` index into a normal column, this is necessary to retain the `Year` as indices are discarded when a column is used as the value to merge with as is done here, i.e. `on='cellref'`

In [ ]:
data_long = data_long.reset_index().merge(xy, on='cellref')
data_long

The resulting dataframe has a value of `pdsi` for every year at every location, with an associated `lat` and `lon` value

## Convert the dataframe to xarray

Now the pandas dataframe can be converted to an `xarray` dataset, which is a well used standard for spatial data analysis. There is a handy function for this purpose, [to_xarray](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_xarray.html).

Some commands are chained before `to_xarray`: 
1. `cellref` column is dropped as it is no longer needed
2. `Year` column to renamed to `year` keep it tidy and case consistent
3. `set_index` creates a multi-index

All the index variables created with `set_index` will automatically be converted to coordinates in the `xarray` object. The order of variables in `set_index` is important: longitude is set to be the fastest varying axis/index, which will result in plots being correctly oriented by default, with longitude on the horizontal axis.

In [ ]:
ds = data_long.drop('cellref', axis=1).rename(columns={'Year': 'year'}).set_index(['year', 'lat', 'lon']).to_xarray()

In [ ]:
ds

Plot as a sanity check using the in-built `plot` method. 

There appears to be a problem: data extended into the Tasman sea from both land masses.

In [ ]:
ds.pdsi.isel(year=50).plot(size = 8);

Looking at the difference between each longitude point and it's neighbour shows the problem: there is difference of -14. at one point, so there is a gap in the coordinates in the Tasman Sea. This is because there are no data points with those longitudes in the original data set.

In [ ]:
ds.coords['lon'] - ds.coords['lon'].shift({'lon':-1})

Fix this by creating a new longitude coordinate that spans the whole range with the same 0.5 degree spacing, and then replace the existing longitude coordinate using [reindex](https://docs.xarray.dev/en/stable/generated/xarray.Dataset.reindex.html)

In [ ]:
ds = ds.reindex(lon = xr.DataArray(np.arange(ds.lon.min(), ds.lon.max()+0.5, 0.5), dims=['lon']))

Now the same plot shows the correct behaviour, with no strange artefacts in the Tasman Sea

In [ ]:
ds.pdsi.isel(year=50).plot(x='lon',y='lat', size = 8);

## Save data for reuse

First step is to add some meta-data. The website says:

> Use Constraints:

> Please cite original publication, online resource, dataset and publication DOIs (where available), and date accessed when using downloaded data. If there is no publication information, please cite investigator, title, online resource, and date accessed. The appearance of external links associated with a dataset does not constitute endorsement by the Department of Commerce/National Oceanic and Atmospheric Administration of external Web sites or the information, products or services contained therein. For other than authorized activities, the Department of Commerce/NOAA does not exercise any editorial control over the information you may find at these locations. These links are provided consistent with the stated purpose of this Department of Commerce/NOAA Web site.

Set the global (dataset level) attributes

In [ ]:
ds.attrs = {
    "title": "Eastern Australia and New Zealand Drought Atlas (ANZDA)",
    "creator_name": "Palmer, J.G.; Cook, E.R.; Turney, C.S.M.; Allen, K.J.; Fenwick, P.; Cook, B.I.; O'Donnell, A.J.; Lough, J.M.; Grierson, P.F.; Baker, P.J.",
    "publisher": "National Centers for Environmental Information, NESDIS, NOAA, U.S. Department of Commerce",
    "references": "Jonathan G. Palmer, Edward R. Cook, Chris S.M. Turney, Kathy Allen, Pavla Fenwick, Benjamin I. Cook, Alison O'Donnell, Janice Lough, Pauline Grierson, Patrick Baker. 2015. Drought variability in the eastern Australia and New Zealand summer drought atlas (ANZDA, CE 1500-2012) modulated by the Interdecadal Pacific Oscillation. Environmental Research Letters, 10(12), 124002. doi:10.1088/1748-9326/10/12/124002",
    "keywords": "Palmer Drought Index Reconstruction, drought, paleoclimate",
    "date_created": "2016-06-06",
    "source": "https://www.ncei.noaa.gov/access/paleo-search/study/20245",
    "source_license": "Please cite original publication, online resource, dataset and publication DOIs (where available), and date accessed when using downloaded data. If there is no publication information, please cite investigator, title, online resource, and date accessed. The appearance of external links associated with a dataset does not constitute endorsement by the Department of Commerce/National Oceanic and Atmospheric Administration of external Web sites or the information, products or services contained therein. For other than authorized activities, the Department of Commerce/NOAA does not exercise any editorial control over the information you may find at these locations. These links are provided consistent with the stated purpose of this Department of Commerce/NOAA Web site.",
    "date_downloaded": datetime.today().strftime('%Y-%m-%d'),
    "geospatial_lat_min": -47.0,
    "geospatial_lat_max": -11.0,
    "geospatial_lon_min": 136.0,
    "geospatial_lon_max": 178.5,
    "time_coverage_start": "450 cal yr BP (1500 CE)",
    "time_coverage_end": "-62 cal yr BP (2012 CE)",
}

Next set variable level attributes

In [ ]:
ds.pdsi.attrs = {
    "name": "Palmer drought severity index",
    "long_name": "Reconstructions of DJF Palmer drought severity index reconstruction series for 1375 Gridpoints",
    "units": "",
}

And coordinate attributes

In [ ]:
ds.lon.attrs = {
    'name': 'longitude',
    'units': 'degrees_east',
}
ds.lat.attrs = {
    'name': 'latitude',
    'units': 'degrees_north',
}

Check the attributes have been specified correctly

In [ ]:
ds

Save to a netcdf file

In [ ]:
ds.to_netcdf("anzda_recon.nc", format = "NETCDF4")

Dump the file header to check it looks ok

In [ ]:
!ncdump -h ~/anzda_recon.nc

Note: The year coordinate remains an integer, and has not been converted to a datetime format. It is possible to do so, but `pandas` does not support datetimes earlier than `1677-09-21 00:12:43.145225` and later than `2262-04-11 23:47:16.854775807`. It is possible to convert to a [cftime index](https://xarray.pydata.org/en/stable/generated/xarray.CFTimeIndex.html), but it was not done for this example. This conversion would be necessary to use time based resampling or groupby.